# Jar SmartSave — Sustainable Savings Intelligence Prototype

**Objective:** Estimate a user's sustainable monthly savings capacity from transaction behaviour and convert it into an actionable savings recommendation.

**Pipeline:** Synthetic Transactions → Monthly Features → Behavioural Features → Target → Time-Based ML → Model Comparison → Recommendation Engine

> **Prototype note:** This notebook uses synthetic data. The safety buffer and target formulation are project assumptions, not actual Jar financial policies.


## 1. Imports

In [ ]:
import pandas as pd  # Imports pandas for tabular data manipulation
import numpy as np  # Imports NumPy for numerical operations and random data generation

from sklearn.linear_model import LinearRegression  # Imports the baseline Linear Regression model
from sklearn.tree import DecisionTreeRegressor  # Imports the Decision Tree regression model
from sklearn.ensemble import RandomForestRegressor  # Imports the Random Forest regression model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Imports model evaluation metrics

from xgboost import XGBRegressor  # Imports the XGBoost regression model


## 2. Create Synthetic Users

In [ ]:
np.random.seed(42)  # Makes random generation reproducible

num_users = 100  # Defines the number of synthetic users

user_ids = [f"U{str(i).zfill(3)}" for i in range(1, num_users + 1)]  # Creates IDs such as U001 to U100

profiles = ["tight", "stable", "surplus"]  # Defines broad financial profiles
income_types = ["stable", "variable"]  # Defines income stability types

user_profiles = []  # Creates an empty list for user profiles

for user in user_ids:  # Loops through every user
    profile = np.random.choice(profiles)  # Randomly assigns a financial profile
    income_type = np.random.choice(income_types, p=[0.7, 0.3])  # Assigns 70% stable and 30% variable income

    if profile == "tight":  # Checks for the tight profile
        base_income = np.random.randint(25000, 40000)  # Assigns a lower income range
    elif profile == "stable":  # Checks for the stable profile
        base_income = np.random.randint(40000, 60000)  # Assigns a medium income range
    else:  # Handles the surplus profile
        base_income = np.random.randint(60000, 90000)  # Assigns a higher income range

    user_profiles.append({  # Stores this user's profile
        "user_id": user,  # Stores the user ID
        "financial_profile": profile,  # Stores the financial profile
        "income_type": income_type,  # Stores income stability
        "base_income": base_income  # Stores approximate monthly income
    })

user_profiles = pd.DataFrame(user_profiles)  # Converts profiles into a DataFrame
user_profiles.head()  # Displays the first few users


## 3. Define Dates and Transaction Categories

In [ ]:
start_date = "2026-03-01"  # Defines the start of the six-month dataset
end_date = "2026-08-31"  # Defines the end of the dataset

transaction_types = {  # Defines transaction categories
    "income": ["Salary", "Freelance"],  # Defines income descriptions
    "essential": ["Rent", "Electricity", "Groceries", "Internet", "Mobile Bill", "Fuel"],  # Defines essential expenses
    "discretionary": ["Swiggy", "Zomato", "Amazon", "Shopping", "Movies", "Dining", "Travel"],  # Defines discretionary expenses
    "savings": ["Jar Savings"]  # Defines savings transactions
}


## 4. Generate Synthetic Transactions

In [ ]:
transactions = []  # Creates an empty list for all transactions

for user in user_ids:  # Loops through every user
    user_info = user_profiles[user_profiles["user_id"] == user].iloc[0]  # Retrieves the current user's profile
    profile = user_info["financial_profile"]  # Stores the financial profile
    income_type = user_info["income_type"]  # Stores the income type
    base_income = user_info["base_income"]  # Stores the base income

    if profile == "tight":  # Defines behaviour for financially tight users
        rent_rate = np.random.uniform(0.30, 0.40)  # Sets rent at 30–40% of income
        grocery_rate = np.random.uniform(0.12, 0.18)  # Sets groceries at 12–18% of income
        utility_rate = np.random.uniform(0.05, 0.08)  # Sets utilities at 5–8% of income
        savings_rate = np.random.uniform(0.03, 0.06)  # Sets savings at 3–6% of income
        discretionary_rate = np.random.uniform(0.08, 0.15)  # Sets discretionary spending at 8–15% of income

    elif profile == "stable":  # Defines behaviour for stable users
        rent_rate = np.random.uniform(0.20, 0.30)  # Sets rent at 20–30% of income
        grocery_rate = np.random.uniform(0.08, 0.13)  # Sets groceries at 8–13% of income
        utility_rate = np.random.uniform(0.04, 0.06)  # Sets utilities at 4–6% of income
        savings_rate = np.random.uniform(0.06, 0.10)  # Sets savings at 6–10% of income
        discretionary_rate = np.random.uniform(0.06, 0.12)  # Sets discretionary spending at 6–12% of income

    else:  # Defines behaviour for surplus users
        rent_rate = np.random.uniform(0.15, 0.25)  # Sets rent at 15–25% of income
        grocery_rate = np.random.uniform(0.06, 0.10)  # Sets groceries at 6–10% of income
        utility_rate = np.random.uniform(0.03, 0.05)  # Sets utilities at 3–5% of income
        savings_rate = np.random.uniform(0.10, 0.18)  # Sets savings at 10–18% of income
        discretionary_rate = np.random.uniform(0.04, 0.10)  # Sets discretionary spending at 4–10% of income

    rent = int(base_income * rent_rate)  # Calculates monthly rent
    monthly_groceries = int(base_income * grocery_rate)  # Calculates monthly groceries
    monthly_utilities = int(base_income * utility_rate)  # Calculates monthly utilities

    for month in pd.date_range(start=start_date, end=end_date, freq="MS"):  # Loops through each month
        if income_type == "stable":  # Checks for stable income
            monthly_income = int(base_income * np.random.uniform(0.97, 1.03))  # Adds small income variation
        else:  # Handles variable income
            monthly_income = int(base_income * np.random.uniform(0.70, 1.30))  # Adds larger income variation

        transactions.append({  # Adds the income transaction
            "user_id": user,  # Stores user ID
            "date": month,  # Stores income date
            "amount": monthly_income,  # Stores income amount
            "description": "Salary",  # Labels income
            "transaction_type": "income"  # Labels transaction type
        })

        transactions.append({  # Adds rent
            "user_id": user,  # Stores user ID
            "date": month + pd.Timedelta(days=1),  # Places rent on day two
            "amount": rent,  # Stores rent amount
            "description": "Rent",  # Labels rent
            "transaction_type": "essential"  # Labels it essential
        })

        transactions.append({  # Adds electricity
            "user_id": user,  # Stores user ID
            "date": month + pd.Timedelta(days=4),  # Places electricity on day five
            "amount": monthly_utilities,  # Stores utility amount
            "description": "Electricity",  # Labels electricity
            "transaction_type": "essential"  # Labels it essential
        })

        grocery_amount = int(monthly_groceries * np.random.uniform(0.90, 1.10))  # Adds grocery variation

        transactions.append({  # Adds groceries
            "user_id": user,  # Stores user ID
            "date": month + pd.Timedelta(days=7),  # Places groceries around day eight
            "amount": grocery_amount,  # Stores grocery amount
            "description": "Groceries",  # Labels groceries
            "transaction_type": "essential"  # Labels it essential
        })

        monthly_discretionary = monthly_income * discretionary_rate  # Calculates expected discretionary spending
        num_discretionary = np.random.randint(5, 12)  # Creates 5–11 discretionary transactions

        for _ in range(num_discretionary):  # Generates each discretionary transaction
            category = np.random.choice(transaction_types["discretionary"])  # Selects a random category
            amount = int(monthly_discretionary / num_discretionary * np.random.uniform(0.5, 1.5))  # Calculates transaction amount
            day = np.random.randint(10, 28)  # Selects a random transaction day

            transactions.append({  # Adds discretionary transaction
                "user_id": user,  # Stores user ID
                "date": month + pd.Timedelta(days=day - 1),  # Creates transaction date
                "amount": amount,  # Stores amount
                "description": category,  # Stores category
                "transaction_type": "discretionary"  # Labels discretionary spending
            })

        monthly_saving_target = monthly_income * savings_rate  # Calculates approximate monthly savings
        num_savings = np.random.randint(4, 8)  # Creates 4–7 savings transactions

        for _ in range(num_savings):  # Generates each savings transaction
            amount = int(monthly_saving_target / num_savings * np.random.uniform(0.7, 1.3))  # Calculates savings amount
            day = np.random.randint(1, 28)  # Selects a savings day

            transactions.append({  # Adds savings transaction
                "user_id": user,  # Stores user ID
                "date": month + pd.Timedelta(days=day - 1),  # Creates savings date
                "amount": amount,  # Stores savings amount
                "description": "Jar Savings",  # Labels savings
                "transaction_type": "savings"  # Labels transaction type
            })


## 5. Create Transaction DataFrame

In [ ]:
df = pd.DataFrame(transactions)  # Converts transactions into a DataFrame
df["date"] = pd.to_datetime(df["date"])  # Converts dates to pandas datetime format
df = df.sort_values(["user_id", "date"]).reset_index(drop=True)  # Sorts transactions chronologically

print("Dataset shape:", df.shape)  # Displays dataset dimensions
print("\nMissing values:")  # Prints a heading
print(df.isnull().sum())  # Checks missing values

df.head()  # Displays sample transactions


## 6. Aggregate Transactions to Monthly Level

In [ ]:
df["month"] = df["date"].dt.to_period("M")  # Creates a month column

monthly_summary = (  # Starts monthly aggregation
    df.groupby(["user_id", "month", "transaction_type"])["amount"]  # Groups by user, month and type
    .sum()  # Sums transaction amounts
    .reset_index()  # Restores normal columns
)

monthly_features = (  # Creates the monthly feature table
    monthly_summary.pivot_table(  # Pivots transaction types into columns
        index=["user_id", "month"],  # Keeps user and month as identifiers
        columns="transaction_type",  # Uses transaction type as columns
        values="amount",  # Uses amount as values
        fill_value=0  # Fills missing categories with zero
    )
    .reset_index()  # Restores normal columns
)

monthly_features.head()  # Displays monthly features


## 7. Core Financial Features

In [ ]:
monthly_features["surplus"] = (  # Creates monthly surplus
    monthly_features["income"]  # Starts with income
    - monthly_features["essential"]  # Subtracts essential expenses
    - monthly_features["discretionary"]  # Subtracts discretionary expenses
)

monthly_features["savings_rate"] = (  # Creates savings rate
    monthly_features["savings"] / monthly_features["income"]  # Divides savings by income
)

monthly_features["savings_rate_pct"] = (  # Creates savings percentage
    monthly_features["savings_rate"] * 100  # Converts rate to percentage
)


## 8. Transaction Count Features

In [ ]:
transaction_counts = (  # Starts calculating transaction counts
    df.groupby(["user_id", "month", "transaction_type"])  # Groups raw transactions
    .size()  # Counts transactions
    .unstack(fill_value=0)  # Pivots types into columns
    .reset_index()  # Restores normal columns
)

transaction_counts = transaction_counts.rename(columns={  # Renames count columns
    "income": "num_income_transactions",  # Renames income count
    "essential": "num_essential_transactions",  # Renames essential count
    "discretionary": "num_discretionary_transactions",  # Renames discretionary count
    "savings": "num_savings_transactions"  # Renames savings count
})

monthly_features = monthly_features.merge(  # Combines counts with financial features
    transaction_counts,  # Uses transaction-count table
    on=["user_id", "month"],  # Matches by user and month
    how="left"  # Keeps all monthly records
)


## 9. Behavioural Feature Engineering

In [ ]:
monthly_features = monthly_features.sort_values(["user_id", "month"]).reset_index(drop=True)  # Ensures chronological order

grouped = monthly_features.groupby("user_id")  # Creates separate history groups for each user

monthly_features["avg_income_3m"] = grouped["income"].transform(lambda x: x.rolling(3, min_periods=1).mean())  # Calculates recent average income
monthly_features["avg_discretionary_3m"] = grouped["discretionary"].transform(lambda x: x.rolling(3, min_periods=1).mean())  # Calculates recent average discretionary spending
monthly_features["avg_savings_3m"] = grouped["savings"].transform(lambda x: x.rolling(3, min_periods=1).mean())  # Calculates recent average savings
monthly_features["avg_surplus_3m"] = grouped["surplus"].transform(lambda x: x.rolling(3, min_periods=1).mean())  # Calculates recent average surplus

monthly_features["income_volatility"] = grouped["income"].transform(lambda x: x.rolling(3, min_periods=1).std())  # Measures income instability
monthly_features["spending_volatility"] = grouped["discretionary"].transform(lambda x: x.rolling(3, min_periods=1).std())  # Measures spending instability
monthly_features["savings_volatility"] = grouped["savings"].transform(lambda x: x.rolling(3, min_periods=1).std())  # Measures savings instability

monthly_features["income_trend"] = grouped["income"].transform(lambda x: x.diff())  # Measures month-over-month income change
monthly_features["spending_growth"] = grouped["discretionary"].transform(lambda x: x.pct_change())  # Measures spending growth
monthly_features["savings_trend"] = grouped["savings"].transform(lambda x: x.diff())  # Measures savings change

monthly_features["expense_to_income"] = (  # Creates total expense burden
    (monthly_features["essential"] + monthly_features["discretionary"]) / monthly_features["income"]  # Divides total expenses by income
)

monthly_features["discretionary_to_income"] = (  # Creates discretionary expense burden
    monthly_features["discretionary"] / monthly_features["income"]  # Divides discretionary spending by income
)

monthly_features["essential_to_income"] = (  # Creates essential expense burden
    monthly_features["essential"] / monthly_features["income"]  # Divides essential spending by income
)

monthly_features = monthly_features.replace([np.inf, -np.inf], np.nan)  # Replaces infinite values with missing values
monthly_features = monthly_features.fillna(0)  # Fills early-history missing values for the prototype


## 10. Create Next-Month Sustainable Savings Target

In [ ]:
monthly_features["next_income"] = monthly_features.groupby("user_id")["income"].shift(-1)  # Retrieves next month's income
monthly_features["next_essential"] = monthly_features.groupby("user_id")["essential"].shift(-1)  # Retrieves next month's essential spending
monthly_features["next_discretionary"] = monthly_features.groupby("user_id")["discretionary"].shift(-1)  # Retrieves next month's discretionary spending
monthly_features["next_savings"] = monthly_features.groupby("user_id")["savings"].shift(-1)  # Retrieves next month's savings

monthly_features["next_surplus"] = (  # Calculates next month's surplus
    monthly_features["next_income"]  # Takes next month's income
    - monthly_features["next_essential"]  # Subtracts next month's essential expenses
    - monthly_features["next_discretionary"]  # Subtracts next month's discretionary expenses
)

monthly_features["future_safe_capacity"] = monthly_features["next_surplus"] * 0.80  # Keeps 80% of future surplus as a conservative capacity estimate
monthly_features["future_safe_capacity"] = monthly_features["future_safe_capacity"].clip(lower=0)  # Prevents negative capacity

monthly_features["sustainable_savings_target"] = (  # Creates the synthetic ML target
    0.7 * monthly_features["future_safe_capacity"]  # Gives 70% weight to safe future capacity
    + 0.3 * monthly_features["next_savings"]  # Gives 30% weight to next-month savings behaviour
)

monthly_features["sustainable_savings_target"] = monthly_features["sustainable_savings_target"].clip(lower=0)  # Prevents negative target values

model_data = monthly_features.dropna(subset=["sustainable_savings_target"]).copy()  # Removes the final month without a future target


## 11. Define Model Features

In [ ]:
feature_columns = [  # Defines all features supplied to the models
    "income",  # Current income
    "essential",  # Current essential spending
    "discretionary",  # Current discretionary spending
    "savings",  # Current savings
    "surplus",  # Current surplus
    "savings_rate",  # Current savings rate
    "income_volatility",  # Recent income instability
    "spending_volatility",  # Recent spending instability
    "savings_volatility",  # Recent savings instability
    "avg_income_3m",  # Three-month income average
    "avg_discretionary_3m",  # Three-month discretionary-spending average
    "avg_savings_3m",  # Three-month savings average
    "avg_surplus_3m",  # Three-month surplus average
    "income_trend",  # Recent income direction
    "spending_growth",  # Recent spending growth
    "savings_trend",  # Recent savings direction
    "expense_to_income",  # Total expense burden
    "discretionary_to_income",  # Discretionary expense burden
    "essential_to_income",  # Essential expense burden
    "num_discretionary_transactions",  # Number of discretionary transactions
    "num_savings_transactions"  # Number of savings transactions
]

X = model_data[feature_columns]  # Creates the feature matrix
y = model_data["sustainable_savings_target"]  # Creates the target vector


## 12. Time-Based Train/Test Split

In [ ]:
train_data = model_data[model_data["month"] <= "2026-06"].copy()  # Uses earlier months for training
test_data = model_data[model_data["month"] > "2026-06"].copy()  # Uses later months for testing

X_train = train_data[feature_columns]  # Selects training features
y_train = train_data["sustainable_savings_target"]  # Selects training target
X_test = test_data[feature_columns]  # Selects test features
y_test = test_data["sustainable_savings_target"]  # Selects test target

print("Training rows:", len(X_train))  # Displays training size
print("Testing rows:", len(X_test))  # Displays testing size


## 13. Train and Evaluate Models

In [ ]:
lr_model = LinearRegression()  # Creates the Linear Regression baseline
lr_model.fit(X_train, y_train)  # Trains Linear Regression
lr_predictions = lr_model.predict(X_test)  # Generates Linear Regression predictions

lr_mae = mean_absolute_error(y_test, lr_predictions)  # Calculates Linear Regression MAE
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))  # Calculates Linear Regression RMSE
lr_r2 = r2_score(y_test, lr_predictions)  # Calculates Linear Regression R-squared

dt_model = DecisionTreeRegressor(max_depth=4, random_state=42)  # Creates a shallow Decision Tree
dt_model.fit(X_train, y_train)  # Trains the Decision Tree
dt_predictions = dt_model.predict(X_test)  # Generates Decision Tree predictions

dt_mae = mean_absolute_error(y_test, dt_predictions)  # Calculates Decision Tree MAE
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_predictions))  # Calculates Decision Tree RMSE
dt_r2 = r2_score(y_test, dt_predictions)  # Calculates Decision Tree R-squared

rf_model = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=3, random_state=42)  # Creates Random Forest
rf_model.fit(X_train, y_train)  # Trains Random Forest
rf_predictions = rf_model.predict(X_test)  # Generates Random Forest predictions

rf_mae = mean_absolute_error(y_test, rf_predictions)  # Calculates Random Forest MAE
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))  # Calculates Random Forest RMSE
rf_r2 = r2_score(y_test, rf_predictions)  # Calculates Random Forest R-squared

xgb_model = XGBRegressor(  # Creates XGBoost
    n_estimators=300,  # Sets number of boosting trees
    max_depth=4,  # Limits tree depth
    learning_rate=0.05,  # Controls each tree's contribution
    subsample=0.8,  # Uses 80% of rows per boosting step
    colsample_bytree=0.8,  # Uses 80% of features per tree
    min_child_weight=3,  # Adds regularization
    reg_lambda=1,  # Adds L2 regularization
    objective="reg:squarederror",  # Specifies regression objective
    random_state=42  # Makes results reproducible
)
xgb_model.fit(X_train, y_train)  # Trains XGBoost
xgb_predictions = xgb_model.predict(X_test)  # Generates XGBoost predictions

xgb_mae = mean_absolute_error(y_test, xgb_predictions)  # Calculates XGBoost MAE
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))  # Calculates XGBoost RMSE
xgb_r2 = r2_score(y_test, xgb_predictions)  # Calculates XGBoost R-squared


## 14. Compare Models

In [ ]:
results = pd.DataFrame({  # Creates a model comparison table
    "Model": ["Linear Regression", "Decision Tree", "Random Forest", "XGBoost"],  # Stores model names
    "MAE": [lr_mae, dt_mae, rf_mae, xgb_mae],  # Stores MAE values
    "RMSE": [lr_rmse, dt_rmse, rf_rmse, xgb_rmse],  # Stores RMSE values
    "R2": [lr_r2, dt_r2, rf_r2, xgb_r2]  # Stores R-squared values
})

results = results.sort_values("MAE").reset_index(drop=True)  # Sorts models by lowest error
results  # Displays the comparison table


## 15. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({  # Creates feature-importance table
    "feature": feature_columns,  # Stores feature names
    "importance": xgb_model.feature_importances_  # Stores XGBoost importance scores
})

feature_importance = feature_importance.sort_values("importance", ascending=False).reset_index(drop=True)  # Sorts by importance
feature_importance.head(15)  # Displays the top 15 features


# Phase 2 — Recommendation Engine

The ML model predicts a sustainable savings amount. The recommendation layer then:

1. Applies a simple safety constraint.
2. Converts the monthly prediction into a daily amount.
3. Compares it with the user's current behaviour.
4. Generates an actionable recommendation.


## 16. Generate Sustainable Savings Prediction

In [ ]:
final_model = xgb_model  # Uses XGBoost as the prototype prediction model

test_data["predicted_sustainable_savings"] = final_model.predict(  # Generates the model prediction
    test_data[feature_columns]  # Supplies the user's financial features
)

test_data["predicted_sustainable_savings"] = test_data["predicted_sustainable_savings"].clip(lower=0)  # Prevents negative recommendations


## 17. Apply Safety Constraint and Convert to Daily Saving

In [ ]:
test_data["predicted_sustainable_savings"] = np.minimum(  # Limits recommendation to current available surplus
    test_data["predicted_sustainable_savings"],  # Uses the ML prediction
    test_data["surplus"].clip(lower=0)  # Uses positive current surplus as the upper limit
)

test_data["recommended_daily_saving"] = test_data["predicted_sustainable_savings"] / 30  # Converts monthly recommendation to daily amount
test_data["current_daily_saving"] = test_data["savings"] / 30  # Converts current monthly savings to daily amount
test_data["recommendation_difference"] = test_data["recommended_daily_saving"] - test_data["current_daily_saving"]  # Measures the change from current behaviour


## 18. Generate Human-Friendly Recommendation

In [ ]:
def generate_recommendation(row):  # Defines a function for converting model output into a product message

    current_daily = row["current_daily_saving"]  # Retrieves current daily saving
    recommended_daily = row["recommended_daily_saving"]  # Retrieves recommended daily saving
    current_monthly = row["savings"]  # Retrieves current monthly savings
    recommended_monthly = row["predicted_sustainable_savings"]  # Retrieves recommended monthly savings

    if recommended_daily > current_daily * 1.10:  # Checks whether savings can increase by more than 10%
        return (  # Returns an increase recommendation
            f"You've been saving around ₹{current_monthly:,.0f} per month. "  # States current behaviour
            f"Based on your recent cash flow, you could comfortably increase "  # Explains the recommendation
            f"that to around ₹{recommended_monthly:,.0f} per month "  # Gives the monthly recommendation
            f"(approximately ₹{recommended_daily:,.0f} per day)."  # Gives the daily recommendation
        )

    elif recommended_daily < current_daily * 0.90:  # Checks whether current saving may be too aggressive
        return (  # Returns a conservative recommendation
            f"You're currently saving around ₹{current_monthly:,.0f} per month. "  # States current behaviour
            f"Based on your recent spending and cash flow, "  # Provides context
            f"a more sustainable level may be around ₹{recommended_monthly:,.0f} per month "  # Gives safer monthly amount
            f"(approximately ₹{recommended_daily:,.0f} per day)."  # Gives safer daily amount
        )

    else:  # Handles users whose current savings are close to the recommendation
        return (  # Returns a maintain recommendation
            f"Your current savings of around ₹{current_monthly:,.0f} per month "  # States current savings
            f"are broadly aligned with your recent cash flow. "  # Explains the result
            f"Continuing at around ₹{recommended_monthly:,.0f} per month "  # Gives recommended monthly amount
            f"(₹{recommended_daily:,.0f} per day) looks reasonable."  # Gives recommended daily amount
        )


## 19. Generate Final Product Output

In [ ]:
test_data["recommendation"] = test_data.apply(  # Applies the recommendation function to every test row
    generate_recommendation,  # Uses the recommendation function
    axis=1  # Applies it row by row
)

recommendation_output = test_data[  # Selects the fields relevant to the product
    [
        "user_id",  # Identifies the user
        "month",  # Identifies the prediction month
        "income",  # Shows current income
        "essential",  # Shows essential spending
        "discretionary",  # Shows discretionary spending
        "savings",  # Shows current savings
        "surplus",  # Shows available surplus
        "predicted_sustainable_savings",  # Shows ML prediction
        "recommended_daily_saving",  # Shows daily recommendation
        "recommendation"  # Shows the human-readable recommendation
    ]
]

recommendation_output.head(10)  # Displays example personalized recommendations


# Prototype Summary

### What we built

- **Synthetic transaction generator** — creates realistic user financial behaviour.
- **Monthly aggregation** — converts transactions into monthly financial signals.
- **Behavioural feature engineering** — captures averages, trends, volatility and spending ratios.
- **Target creation** — estimates sustainable savings using future safe capacity and future savings behaviour.
- **Time-based evaluation** — trains on earlier months and tests on later months.
- **Model comparison** — compares Linear Regression, Decision Tree, Random Forest and XGBoost.
- **Feature importance** — identifies which signals influence the XGBoost model.
- **Recommendation engine** — converts the prediction into a monthly/daily savings recommendation.
- **Safety constraint** — prevents the prototype recommendation from exceeding current positive surplus.
- **Human-readable output** — produces a message that can later be passed to an LLM.

### Final architecture

**Transactions → Features → ML → Sustainable Savings Prediction → Safety Rules → Recommendation → LLM Explanation → Jar UI**

### Important limitation

This is a **proof-of-concept using synthetic data**. The 80% safety buffer and 70/30 target formulation are project assumptions for demonstrating the architecture. A production system would require real transaction data, validated financial policies, stronger risk controls, and extensive offline/online evaluation.
